In [64]:
import nflreadpy as nfl
import pandas as pd

# Load weekly player stats for 2022, 2023, and 2024
player_stats = nfl.load_player_stats(range(2019,2026))

# Convert from Polars to pandas
df = player_stats.to_pandas()

print(df.shape)
print(df.columns.tolist())
print(df.head(10))
print(df.dtypes)

(129812, 150)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving

In [53]:
# What positions exist and how many rows per position
print(df['position'].value_counts())

# Confirm season/week ranges
print(df['season'].unique())
print(df['week'].unique())

# Check for missing values in key fantasy-relevant columns
key_cols = ['player_display_name', 'position', 'team', 'opponent_team',
            'carries', 'targets', 'receptions', 'fantasy_points', 'fantasy_points_ppr']
print(df[key_cols].isnull().sum())

position
WR     7665
LB     7240
CB     6007
RB     4801
DE     4786
DT     4645
TE     3780
SAF    3308
QB     2049
K      1708
P      1687
DB     1587
OT     1410
OLB     964
G       933
FS      889
S       557
MLB     541
C       457
ILB     447
NT      318
FB      290
LS      232
DL       76
OL       14
Name: count, dtype: int64
[2022 2023 2024]
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22]
player_display_name    66
position               66
team                    0
opponent_team           0
carries                 0
targets                 0
receptions              0
fantasy_points          0
fantasy_points_ppr      0
dtype: int64


In [54]:
# Filtering by positions
positions = ['QB', 'RB', 'WR', 'TE', 'K']
df_fantasy = df[df['position'].isin(positions)].copy()

print(df_fantasy.shape)
print(df_fantasy['position'].value_counts())

(20003, 150)
position
WR    7665
RB    4801
TE    3780
QB    2049
K     1708
Name: count, dtype: int64


In [66]:
# Sorting dataset into seasons and rolling averages
df = df.sort_values(['player_id','season','week']).reset_index(drop=True)

# Past 3 game averages
df['target_avg_3'] = df.groupby(['player_id', 'season'])['targets'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_avg_3'] = df.groupby(['player_id', 'season'])['receptions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_yards_avg_3'] = df.groupby(['player_id', 'season'])['receiving_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['target_avg_5'] = df.groupby(['player_id', 'season'])['targets'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_avg_5'] = df.groupby(['player_id', 'season'])['receptions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_yards_avg_5'] = df.groupby(['player_id', 'season'])['receiving_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# player check
player_check = df[
    (df['player_display_name'] == 'Jaxon Smith-Njigba') &
    (df['season'] == 2025)
] [[
    'week',
    'targets',
    'target_avg_3',
    'target_avg_5',
    'receptions',
    'rec_avg_3',
    'rec_avg_5',
    'receiving_yards',
    'rec_yards_avg_3',
    'rec_yards_avg_5'
]]

print(player_check)



        week  targets  target_avg_3  target_avg_5  receptions  rec_avg_3  \
115504     1       13           NaN           NaN           9        NaN   
115505     2       10     13.000000     13.000000           8   9.000000   
115506     3        6     11.500000     11.500000           5   8.500000   
115507     4        5      9.666667      9.666667           4   7.333333   
115508     5        9      7.000000      8.500000           8   5.666667   
115509     6       13      6.666667      8.600000           8   5.666667   
115510     7       14      9.000000      8.600000           8   6.666667   
115511     9        9     12.000000      9.400000           8   8.000000   
115512    10        6     12.000000     10.000000           5   8.000000   
115513    11       12      9.666667     10.200000           9   7.000000   
115514    12       10      9.000000     10.800000           8   7.333333   
115515    13        4      9.333333     10.200000           2   7.333333   
115516    14